# Gemma-2-9B QLoRA — FULL DATA, checkpoint-resumable (TRAINING)

Trains on **all 57K rows** across multiple Kaggle sessions (the 9h limit can't fit one full epoch of 9B on T4 x2). Each session trains up to a time budget, force-saves a checkpoint, and you commit. The next session **attaches the previous output** and resumes.

### Multi-session workflow
1. **Session 1:** Accelerator = **GPU T4 x2**, Internet **on**. Model + competition auto-attach. Save & Run All. It trains ~7.5h, saves checkpoints to `/kaggle/working/ckpt`, commits.
2. From that version's **Output**, create a **Kaggle Dataset** (e.g. `gemma-ckpt`).
3. **Session 2+:** **Add Input** that `gemma-ckpt` dataset. Run again — it auto-detects the latest checkpoint and **resumes**. Commit, then **update the dataset** with the new output.
4. Repeat until it prints `TRAINING COMPLETE` (epoch finished). The final adapter is at `/kaggle/working/adapter` — use it in the `gemma-infer` notebook.

In [ ]:
# Pinned stack + 4-bit kernels (see gemma-train notes). RESTART not needed on a fresh commit;
# on a fresh kernel the install runs before the first `import transformers`.
!pip install -q 'transformers==5.7.0' 'bitsandbytes>=0.46.1'

In [ ]:
# Neutralize huggingface_hub @strict so Gemma2Config imports (version-incompat workaround).
import huggingface_hub, huggingface_hub.dataclasses as _hfd
_noop = lambda cls=None, **kw: (cls if cls is not None else (lambda c: c))
_hfd.strict = _noop
huggingface_hub.strict = _noop
print('patched huggingface_hub.strict')

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch, time
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorWithPadding, TrainerCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import datasets

MAX_LEN = 1024          # competitive context length; lower to 768/512 for more speed
EPOCHS = 1              # one full pass over 57K (spread across sessions)
TRAIN_SECONDS = 27000   # 7.5h train budget per session; leaves buffer under the 9h cap
SAVE_STEPS = 100        # checkpoint frequency (backup against unexpected kills)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:

import json, re, os, glob, numpy as np, pandas as pd

TARGETS = ["winner_model_a", "winner_model_b", "winner_tie"]

def find_dir(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"nothing matched {pattern} under /kaggle/input"
    return os.path.dirname(hits[0])

def parse_list(x):
    if isinstance(x, list): return [str(t) for t in x]
    if not isinstance(x, str): return [""]
    try: v = json.loads(x)
    except Exception: return [x]
    if isinstance(v, list): return ["" if t is None else str(t) for t in v]
    return ["" if v is None else str(v)]

def join(x): return "\n".join(parse_list(x))

def build_text(prompt, resp_a, resp_b, max_chars=6000):
    def clip(s, n):
        s = s or ""
        return s if len(s) <= n else s[: n // 2] + " ... " + s[-n // 2 :]
    return (
        "You are judging which chatbot response a human prefers.\n\n"
        "### Prompt\n" + clip(prompt, max_chars // 3) +
        "\n\n### Response A\n" + clip(resp_a, max_chars // 3) +
        "\n\n### Response B\n" + clip(resp_b, max_chars // 3) +
        "\n\n### Which is preferred? A, B, or tie."
    )


In [ ]:
BASE = find_dir('/kaggle/input/**/config.json')
COMP = find_dir('/kaggle/input/**/train.csv')
print('BASE =', BASE, '\nCOMP =', COMP)

In [ ]:
# FULL data (no subsample). Small stratified val split just for a sanity metric.
df = pd.read_csv(f'{COMP}/train.csv')
df['text'] = [build_text(join(p), join(a), join(b))
              for p, a, b in zip(df['prompt'], df['response_a'], df['response_b'])]
df['label'] = np.argmax(df[TARGETS].values, axis=1)
tr, va = train_test_split(df, test_size=0.02, stratify=df['label'], random_state=42)
print(len(tr), 'train /', len(va), 'val')

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def to_ds(frame):
    ds = datasets.Dataset.from_pandas(frame[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=['text'])
ds_tr, ds_va = to_ds(tr), to_ds(va)

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3, quantization_config=bnb, dtype=torch.float16, device_map='auto')
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)

import bitsandbytes as _bnb
def linear_names(m):
    names = set()
    for n, mod in m.named_modules():
        if isinstance(mod, (torch.nn.Linear, _bnb.nn.Linear4bit)):
            names.add(n.split('.')[-1])
    names -= {'lm_head', 'score', 'classifier'}
    return sorted(names)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='SEQ_CLS', target_modules=linear_names(model))
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Find the latest checkpoint to resume from: prior-session output attached under
# /kaggle/input, or this session's own /kaggle/working/ckpt.
def latest_ckpt():
    hits = glob.glob('/kaggle/input/**/checkpoint-*', recursive=True)
    hits += glob.glob('/kaggle/working/ckpt/checkpoint-*')
    hits = [h for h in hits if os.path.isdir(h) and os.path.exists(h + '/trainer_state.json')]
    return max(hits, key=lambda p: int(p.rstrip('/').split('-')[-1])) if hits else None
RESUME = latest_ckpt()
print('resuming from:', RESUME if RESUME else 'scratch (first session)')

In [ ]:
# Force a save + stop when the per-session time budget is hit, so nothing is lost to the 9h kill.
class TimeStop(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw): self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > TRAIN_SECONDS:
            print(f'[TimeStop] {TRAIN_SECONDS}s budget hit at step {state.global_step} -> saving + stopping')
            control.should_save = True
            control.should_training_stop = True
        return control

def metric(eval_pred):
    logits, labels = eval_pred
    p = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return {'log_loss': log_loss(labels, p, labels=[0,1,2])}

args = TrainingArguments(
    output_dir='/kaggle/working/ckpt', per_device_train_batch_size=1,
    gradient_accumulation_steps=16, per_device_eval_batch_size=1,
    learning_rate=1e-4, num_train_epochs=EPOCHS, warmup_ratio=0.03,
    fp16=True, gradient_checkpointing=True, logging_steps=25,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=2,
    eval_strategy='no', report_to='none', optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metric, callbacks=[TimeStop()])

In [ ]:
out = trainer.train(resume_from_checkpoint=RESUME)
done = trainer.state.global_step >= trainer.state.max_steps
print(f'stopped at step {trainer.state.global_step}/{trainer.state.max_steps} '
      f"-- {'TRAINING COMPLETE' if done else 'partial: resume next session'}")
print('train loss:', out.metrics.get('train_loss'))

In [ ]:
# Always save the current adapter (usable for inference even mid-training).
model.save_pretrained('/kaggle/working/adapter')
tok.save_pretrained('/kaggle/working/adapter')
print(sorted(os.listdir('/kaggle/working/adapter')))
if done:
    val = trainer.evaluate()
    print('VALIDATION log_loss:', val.get('eval_log_loss'))